In [1]:
import pandas as pd
import numpy as np
import math 
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("D:\\skill improvement\\churn data\\data\\processed data\\final_df")


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Age                10000 non-null  int64  
 1   Geography_Germany  10000 non-null  int64  
 2   Balance            10000 non-null  float64
 3   IsActiveMember     10000 non-null  int64  
 4   EstimatedSalary    10000 non-null  float64
 5   NumOfProducts      10000 non-null  int64  
 6   CreditScore        10000 non-null  int64  
 7   Tenure             10000 non-null  int64  
 8   Gender             10000 non-null  int64  
 9   Exited             10000 non-null  int64  
dtypes: float64(2), int64(8)
memory usage: 781.4 KB


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import robust_scale


In [5]:
df.columns

Index(['Age', 'Geography_Germany', 'Balance', 'IsActiveMember',
       'EstimatedSalary', 'NumOfProducts', 'CreditScore', 'Tenure', 'Gender',
       'Exited'],
      dtype='object')

In [6]:
X=df.drop(columns=['Exited'])
y=df['Exited']
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
# printing the split data shape
print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

x_train shape: (8000, 9)
x_test shape: (2000, 9)
y_train shape: (8000,)
y_test shape: (2000,)


In [8]:
df.columns

Index(['Age', 'Geography_Germany', 'Balance', 'IsActiveMember',
       'EstimatedSalary', 'NumOfProducts', 'CreditScore', 'Tenure', 'Gender',
       'Exited'],
      dtype='object')

In [9]:
stand_scale=['Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
ROBUST_SCALE=['CreditScore', 'Age']

In [10]:
# standard scaling
scaler = StandardScaler()
x_train[stand_scale] = scaler.fit_transform(x_train[stand_scale])
x_test[stand_scale] = scaler.transform(x_test[stand_scale])
#robust scaling
x_train[ROBUST_SCALE] = robust_scale(x_train[ROBUST_SCALE])
x_test[ROBUST_SCALE] = robust_scale(x_test[ROBUST_SCALE])

In [11]:
x_train.head()

,Age,Geography_Germany,Balance,IsActiveMember,EstimatedSalary,NumOfProducts,CreditScore,Tenure,Gender
9254,-0.416667,0,-1.218471,1,1.367670,0.808436,0.246269,0.345680,1
1561,0.416667,1,0.696838,1,1.661254,0.808436,-0.156716,-0.348369,1
1670,-1.083333,0,0.618629,0,-0.252807,-0.916688,-0.701493,-0.695393,1
6087,-0.833333,0,0.953212,0,0.915393,-0.916688,-0.686567,1.386753,0
6669,1.583333,0,1.057449,0,-1.059600,-0.916688,-1.014925,1.386753,1


In [12]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier as xgbc

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    "xgboost": xgbc(random_state=42, eval_metric='logloss')
}

results = {}

for name, model in models.items():
    pipeline = ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])
    scores = cross_val_score(pipeline, x_train, y_train, cv=5, scoring='roc_auc')
    results[name] = scores
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Logistic Regression: 0.7594 (+/- 0.0105)
Decision Tree: 0.6979 (+/- 0.0188)
Random Forest: 0.8490 (+/- 0.0056)
xgboost: 0.8419 (+/- 0.0070)
